# Pretraining Dataset Preparation and Model Upscaling

## Introduction

This notebook demonstrates the step-by-step process for preparing datasets and upscaling language models for pretraining tasks. The workflow integrates Hugging Face tools and techniques to preprocess data, combine datasets, and adapt pre-trained models for enhanced performance. By leveraging efficient data cleaning, tokenization, and model upscaling methodologies, the goal is to train a more robust and capable large language model (LLM).

### Key Objectives:
1. **Dataset Preparation**:
   - Combine the `Red Pajama` dataset with Python code from GitHub repositories.
   - Apply advanced cleaning techniques to ensure data quality, including filtering short examples, deduplication, and language checks.
2. **Tokenization and Packaging**:
   - Tokenize and shard the dataset into fixed-length sequences to optimize training efficiency.
3. **Model Configuration and Upscaling**:
   - Start with a pre-trained 12-layer model (`TinySolar-248m-4k`) and upscale its architecture to a 16-layer configuration.
   - Transfer weights from the smaller model while initializing additional layers for the larger model.
4. **Continuous Pretraining**:
   - Configure and train the upscaled model with memory-optimized settings to achieve better performance and scalability.

This project is designed to provide a flexible and reusable workflow for preparing and fine-tuning datasets, adapting model architectures, and pretraining state-of-the-art LLMs.

In [3]:
import warnings
warnings.filterwarnings("ignore")

Here I am going to work with the subset of a much larger dataset called `Red Pajama`. The full set consists of 1 trillion token is available on Hugging Face at [this link](https://huggingface.co/datasets/togethercomputer/RedPajama-Data-1T).

In [4]:
import datasets

pretraining_dataset = datasets.load_dataset(
    "upstage/Pretraining_Dataset",
    split="train"
)

pretraining_dataset = pretraining_dataset.select_columns(
    ["text"]
)

In [5]:
print(pretraining_dataset[0]["text"][:500])

In 1793 Zaman Shah, a grandson of Ahmad Shah Durrani, won a brief war of succession to become ruler of Afghanistan. The support of Painda Khan, chief of the Baraksai branch of the Durrani tribe, was decisive in his victory. In the next fifty year., the brothers of Zaman shah and the sons of Painda Khan were to dominate the affairs of Afghanistan. The Durrani tribe was very large with several branches and numerous clans. 1 Abmad Shah and his successors belonged to the Sadozai clan, but other clan


I just want to download some code and prepare them as a Hugging Face Dataset object to use in training.

This is code pattern is useful as it works for preparing any text scraped from the web.



In [6]:
import os
import requests

code_dir = "./code"

urls = [
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/searches/double_linear_search_recursion.py",
    "https://raw.githubusercontent.com/KosingZhu/tensorflow/master/tensorflow/python/tools/module_util.py",
    "https://raw.githubusercontent.com/EricRemmerswaal/tensorflow/master/tensorflow/python/distribute/distribute_coordinator_context.py",
    "https://raw.githubusercontent.com/computationalartist/tensorflow/master/tensorflow/python/ops/numpy_ops/integration_test/benchmarks/numpy_mlp.py",
    "https://raw.githubusercontent.com/Van-an/tensorflow/master/tensorflow/python/distribute/coordinator/values.py",
    "https://raw.githubusercontent.com/nkgwer/tensorflow/master/tensorflow/lite/tools/visualize.py",
    "https://raw.githubusercontent.com/gitblazer/youtube-dl/master/youtube_dl/version.py",
    "https://raw.githubusercontent.com/PaliC/pytorch/master/test/fx/test_subgraph_rewriter.py"
]

for url in urls:
    print(f"Working on url: {url}")
    response = requests.get(url)
    file_name = os.path.basename(url)
    file_path = os.path.join(code_dir, file_name)

    with open(file_path, "wb") as file:
        file.write(response.content)

Working on url: https://raw.githubusercontent.com/TheAlgorithms/Python/master/searches/double_linear_search_recursion.py
Working on url: https://raw.githubusercontent.com/KosingZhu/tensorflow/master/tensorflow/python/tools/module_util.py
Working on url: https://raw.githubusercontent.com/EricRemmerswaal/tensorflow/master/tensorflow/python/distribute/distribute_coordinator_context.py
Working on url: https://raw.githubusercontent.com/computationalartist/tensorflow/master/tensorflow/python/ops/numpy_ops/integration_test/benchmarks/numpy_mlp.py
Working on url: https://raw.githubusercontent.com/Van-an/tensorflow/master/tensorflow/python/distribute/coordinator/values.py
Working on url: https://raw.githubusercontent.com/nkgwer/tensorflow/master/tensorflow/lite/tools/visualize.py
Working on url: https://raw.githubusercontent.com/gitblazer/youtube-dl/master/youtube_dl/version.py
Working on url: https://raw.githubusercontent.com/PaliC/pytorch/master/test/fx/test_subgraph_rewriter.py


Concatenate scripts into a list:

In [7]:
code_dataset = []
for file in os.listdir(code_dir):
    with open(os.path.join(code_dir, file), 'r', encoding='utf-8', errors='ignore') as f:
        code_dataset.append({'text': f.read()})

In [8]:
code_dataset = datasets.Dataset.from_list(code_dataset)
print(code_dataset)

Dataset({
    features: ['text'],
    num_rows: 9
})


Combine the python code dataset with the pretraining dataset you downloaded above:

In [9]:
dataset = datasets.concatenate_datasets(
    [pretraining_dataset, code_dataset]
)
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 60009
})


## Data Cleaning

1. Filter out samples that are too short
2. Remove repetitions within a single text example
3. Remove duplicated documents
4. Quality filter to remove non-English texts

### Remove examples that are too short

In [10]:
import heapq

def paragraph_length_filter(x):
    """Returns False iff a page has too few lines or lines are too short."""
    lines = x['text'].split('\n')
    if (
        len(lines) < 3
        or min(heapq.nlargest(3, [len(line) for line in lines])) < 3
    ):
        return False
    return True

In [11]:
dataset = dataset.filter(
    paragraph_length_filter,
    load_from_cache_file=False
)

Filter:   0%|          | 0/60009 [00:00<?, ? examples/s]

### Remove repeated text within training examples

In [12]:
def find_duplicates(paragraphs):
    """
    Use this function to find the number of repetitions
    in the paragraphs.
    """
    unique_x = set()
    duplicate_chars = 0
    duplicate_elements = 0
    for element in paragraphs:
        if element in unique_x:
            duplicate_chars += len(element)
            duplicate_elements += 1
        else:
            unique_x.add(element)
    return duplicate_elements, duplicate_chars

In [13]:
import re

def paragraph_repetition_filter(x):
    """
    Returns False iff a page has too many repetitions.
    """
    text = x['text']
    paragraphs = re.compile(r"\n{2,}").split(text.strip())                # Split by paragraphs (2 or more newlines)
    paragraphs_duplicates, char_duplicates = find_duplicates(paragraphs)  # Find number of duplicates in paragraphs
    if paragraphs_duplicates / len(paragraphs) > 0.3:
        return False
    if char_duplicates / len(text) > 0.2:
        return False
    return True

In [14]:
dataset = dataset.filter(
    paragraph_repetition_filter,
    load_from_cache_file=False
)

Filter:   0%|          | 0/52357 [00:00<?, ? examples/s]

### Remove Deduplication

In [15]:
def deduplication(ds):
    def dedup_func(x):
        """Use this function to remove duplicate entries"""
        if x['text'] in unique_text:
            return False
        else:
            unique_text.add(x['text'])
            return True

    unique_text = set()

    ds = ds.filter(dedup_func, load_from_cache_file=False, num_proc=1)
    return ds

dataset = deduplication(dataset)

Filter:   0%|          | 0/52327 [00:00<?, ? examples/s]

### Quality filter - Language

Remove any text examples that are in a language other than English. The code here uses a language detection model called fastText.

In [16]:
import urllib
from fasttext.FastText import _FastText

def english_language_filter(ds):
    # load language detection model
    model = _FastText("./models/L2_language_model.bin")

    def is_english(x):
        # Predict language of the text and probability
        language, score = model.predict(x['text'].replace("\n", ""))

        language = language[0].split("__")[2]
        return score > 0.4 and language == "en" # change code here if building a model in another language

    ds = ds.filter(is_english, load_from_cache_file=False, num_proc=1)
    return ds

dataset = english_language_filter(dataset)

Parameter 'function'=<function english_language_filter.<locals>.is_english at 0x1271caa20> of the transform datasets.arrow_dataset.Dataset.filter@2.0.1 couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Filter:   0%|          | 0/43598 [00:00<?, ? examples/s]

In [17]:
dataset.num_rows

40473

In [18]:
# Svae the dataset as a parquet file
file_path = "./data/preprocessed_dataset.parquet"
dataset.to_parquet(file_path)

Creating parquet from Arrow format:   0%|          | 0/41 [00:00<?, ?ba/s]

197100832

## Packaging the Dataset for Pre-training

In [19]:
dataset = datasets.load_dataset(
    "parquet",
    data_files="./data/preprocessed_dataset.parquet",
    split="train"
)
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 40473
})


I used the `shard` method of the Hugging Face `Dataset` object to split the dataset into 10 smaller pieces. See details in [shard documentation](https://huggingface.co/docs/datasets/en/process#shard).

The number of rows should be 4048 after shardding.

In [20]:
dataset = dataset.shard(num_shards=10, index=0)
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 4048
})


Then, load a tokenizer so that the sharded dataset can be tokenized for packaging

In [ ]:
from transformers import AutoTokenizer
model_path_or_name = "Upstage/SOLAR-10.7B-v1.0"
tokenizer = AutoTokenizer.from_pretrained(
    model_path_or_name,
    device_map="auto",
    use_fast=False
)

# Save model and tokenizer locally for future use
tokenizer.save_pretrained("./models/SOLAR-10.7B-v1.0")

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

In [22]:
tokenizer.tokenize("I'm a short sentence")

['▁I', "'", 'm', '▁a', '▁short', '▁sentence']

In [23]:
# Helper function to tokenize the dataset

def tokenization(example):
    # Tokenize
    tokens = tokenizer.tokenize(example["text"])

    # Convert tokens to ids
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    # Add <bos>, <eos> tokens to the front and back of tokens_ids
    # bos: begin of sequence, eos: end of sequence
    token_ids = [
        tokenizer.bos_token_id] \
        + token_ids \
        + [tokenizer.eos_token_id
    ]
    example["input_ids"] = token_ids

    # We will be using this column to count the total number of tokens
    # in the final dataset
    example["num_tokens"] = len(token_ids)
    return example

In [24]:
dataset = dataset.map(tokenization, load_from_cache_file=False)
print(dataset)

Map:   0%|          | 0/4048 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'num_tokens'],
    num_rows: 4048
})


1. Concatenate input_ids for all examples into a single list
2. Discard extra tokens from end of the list so number of tokens is exactly divisible by `max_seq_length`
3. Reshape so the length equals to `max_seq_length`

In [25]:
import numpy as np
input_ids = np.concatenate(dataset["input_ids"])
print(len(input_ids))

4924245


In [26]:
max_seq_length = 32
total_length = len(input_ids) - len(input_ids) % max_seq_length
print(total_length)

4924224


In [27]:
input_ids = input_ids[:total_length]
print(input_ids.shape)

(4924224,)


In [28]:
input_ids_reshaped = input_ids.reshape(-1, max_seq_length).astype(np.int32)
input_ids_reshaped.shape

(153882, 32)

In [29]:
input_ids_list = input_ids_reshaped.tolist()
packaged_pretrain_dataset = datasets.Dataset.from_dict(
    {"input_ids": input_ids_list}
)
print(packaged_pretrain_dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 153882
})


In [30]:
# Save packed dataset
packaged_pretrain_dataset.to_parquet("./data/packaged_pretrain_dataset.parquet")

Creating parquet from Arrow format:   0%|          | 0/154 [00:00<?, ?ba/s]

20312424

## Model Configuration for Pre-training

Here I am working with a general pre-trained `Upstage/TinySolar-248m-4k` model. I decided to upscale this model from 12 layers to 16 layers

1. Configure a 16 layer model and initialize it with random weights
2. Load the 12 layer tinySolar-248m-4k model into memory
3. Copy the bottom 8 and top 8 layers from the 12 layer model and use them to overwrite the random weights of the 16 layer model
4. Copy over the embedding and classifying layers to replace the randomly initialized counterparts in the 16 layer model

Start by creating a LlamaConfig object to configure the architecture of the model:

In [32]:
def print_nparams(model):
    """Calculate the total number of model parameters"""
    nparams = sum(p.numel() for p in model.parameters())
    print(f"The total number of parameters is: {nparams}")

In [31]:
from transformers import LlamaConfig
config = LlamaConfig()
print(config)

LlamaConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "transformers_version": "4.47.1",
  "use_cache": true,
  "vocab_size": 32000
}



In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Upstage/TinySolar-248m-4k")
tokenizer = AutoTokenizer.from_pretrained("Upstage/TinySolar-248m-4k")

model.save_pretrained("./models/TinySolar-248m-4k")
tokenizer.save_pretrained("./models/TinySolar-248m-4k")

('./models/TinySolar-248m-4k/tokenizer_config.json',
 './models/TinySolar-248m-4k/special_tokens_map.json',
 './models/TinySolar-248m-4k/tokenizer.model',
 './models/TinySolar-248m-4k/added_tokens.json',
 './models/TinySolar-248m-4k/tokenizer.json')

In [34]:
# NOTE: Running large models in a limited environment. Run me if you encounter any memory issues.
import gc
del model
gc.collect()

369

In [35]:
import torch
from transformers import AutoModelForCausalLM

model_name_or_path = "./models/TinySolar-248m-4k"
pretrained_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print_nparams(pretrained_model)  # 248013824 => 248M

The total number of parameters is: 248013824


Following is the config of our 16 layers model:

In [36]:
config = LlamaConfig(
    num_hidden_layers=16,
    hidden_size=1024,
    intermediate_size=4096,
    num_attention_heads=32,
    num_key_value_heads=8,
    torch_dtype="bfloat16",
    use_cache=False
)
print(config)

LlamaConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 32,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.47.1",
  "use_cache": false,
  "vocab_size": 32000
}



In [ ]:
from transformers import LlamaForCausalLM
model = LlamaForCausalLM(config)
model = model.to(dtype=torch.bfloat16)  # convert to float16
print_nparams(model)  # 308839424 => 308M

The total number of parameters is: 308839424


Copy the bottom 8 and top 8 layers from the 12 layer model and use them to overwrite the random weights of the 16 layer model

In [38]:
from copy import deepcopy

model.model.layers = deepcopy(pretrained_model.model.layers[:-4]) \
    + deepcopy(pretrained_model.model.layers[4:])

model.model.embed_tokens = deepcopy(pretrained_model.model.embed_tokens)

model.lm_head = deepcopy(pretrained_model.lm_head)

print(model.config)

LlamaConfig {
  "_attn_implementation_autoset": true,
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 32,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.47.1",
  "use_cache": false,
  "vocab_size": 32000
}



In [39]:
print_nparams(model)  # 308839424 => 308M

The total number of parameters is: 308839424


In [40]:
model.save_pretrained('./models/TinySolar-308m-4k-init')

In [41]:
del pretrained_model
del model
gc.collect()

228

### Model Training with Continuous Pre-training

In [42]:
pretrained_model = AutoModelForCausalLM.from_pretrained(
    "./models/TinySolar-308m-4k-init",
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_cache=False,
)

pretrained_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 1024)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1024, out_features=256, bias=False)
          (v_proj): Linear(in_features=1024, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=1024, out_features=4096, bias=False)
          (up_proj): Linear(in_features=1024, out_features=4096, bias=False)
          (down_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((1024,), eps=1e-06)
      )
    )
    (norm): 

In [43]:
# Load Dataset
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, args, split="train"):
        """Initializes the custom dataset object."""
        self.args = args
        self.dataset = datasets.load_dataset(
            "parquet",
            data_files=args.dataset_name,
            split=split
        )

    def __len__(self):
        """Returns the number of samples in the dataset."""
        return len(self.dataset)

    def __getitem__(self, idx):
        """
        Retrieves a single data sample from the dataset
        at the specified index
        """
        # Convert the lists to a LongTensor for PyTorch
        input_ids = torch.LongTensor(self.dataset[idx]["input_ids"])
        labels = torch.LongTensor(self.dataset[idx]["input_ids"])

        # Return the sample as a dictionary
        return {"input_ids": input_ids, "labels": labels}

Configure Training Arguments:

In [44]:
from dataclasses import dataclass, field
import transformers

@dataclass
class CustomArguments(transformers.TrainingArguments):
    dataset_name: str = field(                           # Dataset configuration
        default="./data/packaged_pretrain_dataset.parquet")
    num_proc: int = field(default=1)                     # Number of subprocesses for data preprocessing
    max_seq_length: int = field(default=32)              # Maximum sequence length

    # Core training configurations
    seed: int = field(default=0)                         # Random seed for initialization, ensuring reproducibility
    optim: str = field(default="adamw_torch")            # Optimizer, here it's AdamW implemented in PyTorch
    max_steps: int = field(default=30)                   # Number of maximum training steps
    per_device_train_batch_size: int = field(default=2)  # Batch size per device during training

    # Other training configurations
    learning_rate: float = field(default=5e-5)           # Initial learning rate for the optimizer
    weight_decay: float = field(default=0)               # Weight decay
    warmup_steps: int = field(default=10)                # Number of steps for the learning rate warmup phase
    lr_scheduler_type: str = field(default="linear")     # Type of learning rate scheduler
    gradient_checkpointing: bool = field(default=True)   # Enable gradient checkpointing to save memory
    dataloader_num_workers: int = field(default=2)       # Number of subprocesses for data loading
    bf16: bool = field(default=True)                     # Use bfloat16 precision for training on supported hardware
    gradient_accumulation_steps: int = field(default=1)  # Number of steps to accumulate gradients before updating model weights

    # Logging configuration
    logging_steps: int = field(default=3)                # Frequency of logging training information
    report_to: str = field(default="none")               # Destination for logging (e.g., WandB, TensorBoard)

    # Saving configuration
    # save_strategy: str = field(default="steps")          # Can be replaced with "epoch"
    # save_steps: int = field(default=3)                   # Frequency of saving training checkpoint
    # save_total_limit: int = field(default=2)             # The total number of checkpoints to be saved

Parse the custom arguments and set the output directory where the model will be saved:

In [45]:
parser = transformers.HfArgumentParser(CustomArguments)
args, = parser.parse_args_into_dataclasses(
    args=["--output_dir", "output"]
)

Setup the training dataset:

In [46]:
train_dataset = CustomDataset(args=args)

Generating train split: 0 examples [00:00, ? examples/s]

Set up a callback to log the loss values during training

In [47]:
from transformers import TrainerCallback

# Define a custom callback to log the loss values
class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            self.logs.append(logs)

    def __init__(self):
        self.logs = []

# Initialize the callback
loss_logging_callback = LossLoggingCallback()

In [48]:
import multiprocessing
multiprocessing.set_start_method("fork", force=True)

Create an instance of the Hugging Face Trainer object from the transformers library. Call the train() method of the trainder to initialize the training run:

In [49]:
from transformers import Trainer

trainer = Trainer(
    model=pretrained_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=None,
    callbacks=[loss_logging_callback]
)

trainer.train()

  0%|          | 0/30 [00:00<?, ?it/s]

{'loss': 4.8987, 'grad_norm': 36.75, 'learning_rate': 1.5e-05, 'epoch': 0.0}
{'loss': 4.6152, 'grad_norm': 50.0, 'learning_rate': 3e-05, 'epoch': 0.0}
{'loss': 5.1772, 'grad_norm': 42.0, 'learning_rate': 4.5e-05, 'epoch': 0.0}
{'loss': 4.2914, 'grad_norm': 43.75, 'learning_rate': 4.5e-05, 'epoch': 0.0}
{'loss': 4.2466, 'grad_norm': 48.5, 'learning_rate': 3.7500000000000003e-05, 'epoch': 0.0}
{'loss': 3.8507, 'grad_norm': 33.0, 'learning_rate': 3e-05, 'epoch': 0.0}
{'loss': 4.4585, 'grad_norm': 42.0, 'learning_rate': 2.25e-05, 'epoch': 0.0}
{'loss': 4.4742, 'grad_norm': 43.75, 'learning_rate': 1.5e-05, 'epoch': 0.0}
{'loss': 3.9659, 'grad_norm': 47.25, 'learning_rate': 7.5e-06, 'epoch': 0.0}
{'loss': 4.5628, 'grad_norm': 40.75, 'learning_rate': 0.0, 'epoch': 0.0}
{'train_runtime': 14.2177, 'train_samples_per_second': 4.22, 'train_steps_per_second': 2.11, 'train_loss': 4.454104105631511, 'epoch': 0.0}


TrainOutput(global_step=30, training_loss=4.454104105631511, metrics={'train_runtime': 14.2177, 'train_samples_per_second': 4.22, 'train_steps_per_second': 2.11, 'total_flos': 3180342804480.0, 'train_loss': 4.454104105631511, 'epoch': 0.0003899091511677779})

In [50]:
pretrained_model.save_pretrained("./models/upstage/output/checkpoint-10000")

In [51]:
from transformers import AutoTokenizer, TextStreamer, AutoModelForCausalLM
import torch

model_name_or_path = "./models/upstage/output/checkpoint-10000"
model2 = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained("./models/TinySolar-248m-4k")

In [63]:
from transformers import LogitsProcessorList, TemperatureLogitsWarper, InfNanRemoveLogitsProcessor
prompt = "I am an engineer. I love"

# Ensure the model and inputs are on the correct device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model2.to(device)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True
)

outputs = model2.generate(
    **inputs,
    streamer=streamer,
    use_cache=True,
    max_new_tokens=64,
    do_sample=True,
    temperature=1.0
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


that. The most helpful!
Takes a little different from this because it starts my career at a big branding company. This person just says they got a text. It't been for that way, it'll come out and if youve got one, then it would seem to be completely fair.
